# 📑 Day-4 · Notebook 01: Document Loaders & Chunking Strategies
**KIT Campus RAG Agent · Part 1: Ingestion, Multi-Format Parsing & Chunk Physics**

Welcome to Day 4! Today we build production **Retrieval-Augmented Generation (RAG)** systems and **LangGraph AI Agents** for **KIT College of Engineering & Technology**.

In this notebook, we cover:
1. **Multi-Format Document Parsing**: Ingesting PDF (`pypdf`), Word (`python-docx`), Markdown (`.md`), and Tabular CSV (`.csv`) into standard LangChain `Document` schemas.
2. **Chunking Strategies**: Fixed-size vs Recursive Character Text Splitter (`RecursiveCharacterTextSplitter`).
3. **The Physics of Chunk Overlap (10%–20%)**: Preventing context tearing across sentences and policy clauses.
4. **Metadata Preservation**: Tracking `source`, `section`, `format`, and `lineage` for reliable citations.



In [ ]:
import os
import sys
from typing import List, Dict, Any
from dataclasses import dataclass, field

# Standard LangChain Document structure
try:
    from langchain_core.documents import Document
    print("✓ Successfully imported Document from langchain_core.documents")
except ImportError:
    @dataclass
    class Document:
        page_content: str
        metadata: Dict[str, Any] = field(default_factory=dict)
    print("⚠ Using fallback Document dataclass")



### 1. Ingesting Multi-Format Student Documents
In a campus RAG agent, student questions touch different file types:
- **Markdown (.md)**: Academic Regulations, Attendance Condonation, CIA Rules (`kit_college_student_handbook.md`)
- **Word (.docx)**: Hostel Curfew (8:30 PM), Mess Timings, Wi-Fi Policy (`kit_campus_hostel_and_mess_guide.docx`)
- **PDF (.pdf)**: Placement CGPA Cutoff (7.0), Dream Company CTC (10 LPA) (`kit_placement_policy_guide.pdf`)
- **Tabular (.csv)**: Course Codes (CS401), Faculty Names, Exam Schedules (`kit_courses_and_curriculum.csv`)

Let's inspect how to parse these cleanly.



In [ ]:
# Path to the data folder
DATA_DIR = os.path.abspath(os.path.join("..", "01_local_first_principles_rag", "data"))
if not os.path.exists(DATA_DIR):
    DATA_DIR = os.path.abspath(os.path.join(".", "data"))

print(f"Data directory: {DATA_DIR}")
if os.path.exists(DATA_DIR):
    print("Files found:", os.listdir(DATA_DIR))
else:
    print("Data directory path will be resolved during full execution.")



### 2. LangChain RecursiveCharacterTextSplitter
Why not naive character splitting?
- **Naive Splitting**: Cuts words in half (e.g., `condo-` / `nation`) or separates rules from their conditions.
- **Recursive Splitting**: Tries natural paragraph boundaries (`\n\n`), then headers (`\n## `), sentences (`\n`, `. `), and words (` `).



In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

sample_student_handbook_text = """## 1. Academic Regulations & Attendance Policy
Every enrolled student must maintain a minimum cumulative attendance of 75% in each registered course to be eligible for Semester End Examinations (SEE).

### 1.2 Medical Condonation Policy (65% to 74.9%)
Students whose attendance falls between 65.0% and 74.9% due to genuine medical illness may apply for Medical Condonation.
Required Procedure:
1. Submit a valid Government-registered Medical Practitioner Certificate within 3 calendar days of resuming classes.
2. Pay a non-refundable condonation processing fee of ₹500 per subject.
3. Final condonation is approved by the Dean of Academic Affairs.

### 1.3 Detention Rule (<65%)
Any student with attendance below 65.0% in any course shall be awarded the grade 'DE' (Detained) and barred from SEE.
"""

splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=50,
    separators=["\n## ", "\n### ", "\n\n", "\n", ". ", " "]
)

chunks = splitter.split_text(sample_student_handbook_text)
print(f"Original Text Length: {len(sample_student_handbook_text)} chars")
print(f"Generated Chunks: {len(chunks)}\n")

for i, chunk in enumerate(chunks, 1):
    print(f"--- Chunk {i} ({len(chunk)} chars) ---")
    print(chunk.strip())
    print()



### 3. Chunk Overlap Physics (Preventing Boundary Context Tearing)
Notice how Chunk 1 and Chunk 2 share boundary words.
If a student asks: *"What is the condonation fee if my attendance is 68%?"*, overlap ensures the condition (`65% to 74.9%`) and the action (`fee of ₹500`) are not severed into disconnected chunks!

